# Финансовое состояние малых и средних строительных компаний

ОКВЭД 41.20

Источники: ГИР БО ФНС и Реестр МСП ФНС.


In [ ]:
!pip -q install geopandas geopy

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

data_zip = Path('/content/stroitelstvo_data.zip')

if not data_zip.exists():
    files.upload()

work = Path('/content/data_4120')
work.mkdir(exist_ok=True)

with zipfile.ZipFile(data_zip) as z:
    z.extractall(work)


## Проверка данных


In [ ]:
cohort = pd.read_csv(work / 'cohort_4120.csv', dtype={'inn': str, 'okved': str}).fillna('')
status = pd.read_csv(work / 'collection_status.csv', dtype={'inn': str})
raw = pd.read_csv(work / 'company_year_financials.csv.gz', dtype={'inn': str})

assert len(cohort) == 8927 and cohort['inn'].is_unique
assert cohort['category'].isin(['Малое предприятие', 'Среднее предприятие']).all()
assert cohort['okved'].eq('41.20').all()
assert raw['inn'].isin(cohort['inn']).all()

print(f'В выборке {len(cohort)} компаний, отчётных строк {len(raw)}')
print(f'Обработано ИНН: {len(status)} из {len(cohort)}')
display(status['status'].value_counts())
display(raw[['inn', 'year', 'revenue_thousand_rub', 'net_profit_thousand_rub']].head())

## Финансовые показатели


In [ ]:
raw = raw.sort_values('published_correction_date').drop_duplicates(['inn', 'year'], keep='last').copy()

money_cols = [c for c in raw.columns if c.endswith('_thousand_rub')]
raw[money_cols] = raw[money_cols].apply(pd.to_numeric, errors='coerce')

raw['ebit'] = raw['profit_before_tax_thousand_rub'] + raw['interest_expense_thousand_rub']

annual = raw.groupby('year').agg(
    companies=('inn', 'nunique'),
    revenue=('revenue_thousand_rub', 'sum'),
    profit=('net_profit_thousand_rub', 'sum'),
    profitable=('net_profit_thousand_rub', lambda x: (x > 0).sum()),
    profit_count=('net_profit_thousand_rub', 'count'),
    ebit=('ebit', 'sum')
).reset_index()

annual['profitable_pct'] = 100 * annual['profitable'] / annual['profit_count']
display(annual)

plt.figure(figsize=(9, 4))
plt.plot(annual['year'], annual['profit'] / 1e6, marker='o', label='Чистая прибыль')
plt.plot(annual['year'], annual['ebit'] / 1e6, marker='o', label='EBIT')
plt.xlabel('Год')
plt.ylabel('млрд руб.')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Изменение чистой прибыли


In [ ]:
profit = raw[['inn', 'year', 'net_profit_thousand_rub']].copy()

prev = profit.copy()
prev['year'] = prev['year'] + 1
prev = prev.rename(columns={'net_profit_thousand_rub': 'prev_profit'})

growth = profit.merge(prev, on=['inn', 'year'])
growth = growth[growth['prev_profit'] > 0].copy()
growth['growth_pct'] = 100 * (growth['net_profit_thousand_rub'] / growth['prev_profit'] - 1)

summary = growth.groupby('year').agg(
    companies=('inn', 'count'),
    median_growth=('growth_pct', 'median'),
    decreased=('growth_pct', lambda x: (x < 0).sum())
).reset_index()

summary['decreased_pct'] = 100 * summary['decreased'] / summary['companies']
display(summary)

year = int(growth['year'].max())
x = growth.loc[growth['year'] == year, 'growth_pct']
x = x[x.between(-200, 200)]

plt.figure(figsize=(10, 4))
plt.hist(x, bins=np.arange(-200, 210, 10), edgecolor='black')
plt.xlabel('Изменение прибыли, %')
plt.ylabel('Количество компаний')
plt.title(f'{year} к {year-1}')
plt.grid(axis='y', alpha=0.2)
plt.show()


## Топ-500 компаний по выручке


In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

year = raw['year'].max()
top = raw[(raw['year'] == year) & raw['revenue_thousand_rub'].notna()]
top = top.sort_values('revenue_thousand_rub', ascending=False).head(500).copy()

top[['settlement', 'city', 'district', 'region']] = top[['settlement', 'city', 'district', 'region']].fillna('')

def address(row):
    place = row['settlement'] or row['city'] or row['district']
    return ', '.join([x for x in [place, row['region'], 'Россия'] if x])

top['address'] = top.apply(address, axis=1)

geolocator = Nominatim(user_agent='study_project')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

coords = {}

for a in top['address'].unique():
    loc = geocode(a, country_codes='ru')
    if loc is not None:
        coords[a] = (loc.latitude, loc.longitude)

top['latitude'] = top['address'].map(lambda x: coords.get(x, (np.nan, np.nan))[0])
top['longitude'] = top['address'].map(lambda x: coords.get(x, (np.nan, np.nan))[1])

points = top.dropna(subset=['latitude', 'longitude']).copy()

plt.figure(figsize=(12, 6))
plt.scatter(points['longitude'], points['latitude'], s=15, alpha=0.6)
plt.xlim(20, 180)
plt.ylim(40, 82)
plt.xlabel('Долгота')
plt.ylabel('Широта')
plt.title(f'Топ-500 компаний по выручке, {year}')
plt.grid(alpha=0.2)
plt.show()


## Итоги


In [ ]:
last = annual.iloc[-1]
year = int(last['year'])

print('Год:', year)
print('Количество компаний:', int(last['companies']))
print('Доля прибыльных компаний:', round(last['profitable_pct'], 1), '%')

last_growth = summary.iloc[-1]
print('Медианное изменение чистой прибыли:', round(last_growth['median_growth'], 1), '%')
print('Доля компаний со снижением прибыли:', round(last_growth['decreased_pct'], 1), '%')
